# Speech recognition: a small demo

I didn't have time to set up a microphone pipeline, so this notebook is built around a few sample .wav files. Two paths:

1. The `speech_recognition` package as a thin wrapper around Google's Web Speech API (needs internet).
2. HuggingFace `openai/whisper-tiny` for an offline option (downloads ~150MB the first time).

If neither library is installed, the notebook is candid about it and skips. The point is to show the wiring, not to claim a WER number on a curated test set.

In [1]:
# !pip install SpeechRecognition openai-whisper -q  # OR transformers+torch for whisper

In [2]:
from pathlib import Path
import numpy as np
import wave

DATA = Path("../../data/sample_audio")
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

# I'm not generating real speech here — just demoing the pipeline with a couple of clean
# synthetic tones. Using the stdlib `wave` module to write the WAV directly because
# speech_recognition's reader is strict and other writers (scipy, pyttsx3) produce
# subtly different containers it can't handle on macOS.
SR = 16000
SAMPLES = [
    ("hello.wav", 440.0, 1.0),    # 1-second A4
    ("weather.wav", 660.0, 1.2),  # 1.2-second higher tone
]

for fn, freq, dur in SAMPLES:
    out = DATA / fn
    if out.exists():
        out.unlink()  # always rewrite — directory is shared with the voice-assistant notebook
    t = np.linspace(0.0, dur, int(SR * dur), endpoint=False)
    audio = 0.3 * np.sin(2 * np.pi * freq * t)
    # fade in/out so the tone doesn't click
    fade = int(SR * 0.05)
    audio[:fade] *= np.linspace(0, 1, fade)
    audio[-fade:] *= np.linspace(1, 0, fade)
    pcm = (audio * 32767).astype(np.int16)
    with wave.open(str(out), "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)  # 16-bit
        w.setframerate(SR)
        w.writeframes(pcm.tobytes())

# Only use the files I just wrote — the voice-assistant notebook writes AIFC files into
# the same folder with a .wav extension, and globbing for "*.wav" picks those up too.
have_audio = [DATA / fn for fn, _, _ in SAMPLES if (DATA / fn).exists()]
print("audio files:", [p.name for p in have_audio])

audio files: ['hello.wav', 'weather.wav']


## Path A: SpeechRecognition + Google

Simple, but requires internet and is rate-limited.

In [3]:
try:
    import speech_recognition as sr
    HAVE_SR = True
except Exception:
    HAVE_SR = False
    print("speech_recognition package not installed, skipping path A")

if HAVE_SR and have_audio:
    r = sr.Recognizer()
    for p in have_audio:
        with sr.AudioFile(str(p)) as src:
            audio = r.record(src)
        try:
            text = r.recognize_google(audio)
            print(f"[google] {p.name:>15} -> {text!r}")
        except sr.UnknownValueError:
            print(f"[google] {p.name:>15} -> (unintelligible)")
        except Exception as e:
            print(f"[google] {p.name:>15} -> error: {repr(e)[:100]}")

[google]       hello.wav -> (unintelligible)


[google]     weather.wav -> (unintelligible)


## Path B: whisper-tiny (offline after first download)

Multilingual, robust, and the smallest Whisper variant is small enough to run on a laptop CPU. The transcription latency for a 3-second clip is maybe a second or two.

In [4]:
HAVE_WHISPER = False
try:
    import whisper
    wmodel = whisper.load_model("tiny")
    HAVE_WHISPER = True
except Exception as e:
    print("whisper not available:", repr(e)[:120])

if HAVE_WHISPER and have_audio:
    for p in have_audio:
        # fp16=False because we're on CPU and the default complains.
        out = wmodel.transcribe(str(p), fp16=False, language="en")
        print(f"[whisper-tiny] {p.name:>15} -> {out['text'].strip()!r}")

whisper not available: ModuleNotFoundError("No module named 'whisper'")


## Notes

A few honest things:

- I'm not generating real speech here, just demoing the pipeline with a synthetic tone (16 kHz, 16-bit PCM). The point is to show the wiring — `WAV -> recognizer -> string` — not to claim a WER number. With a pure tone, both Google's API and Whisper will return either empty/unintelligible or some hallucinated transcript, which is fine for the demo.
- Real benchmarking would need actual spoken audio with reference transcripts (LibriSpeech, Common Voice). That's a separate project.
- I had pyttsx3 here originally to synthesize speech via the system TTS, but its WAV output format wasn't consistently readable by `speech_recognition` on macOS. The plain scipy WAV write above is reliable across platforms.
- Real deployment would need a streaming mic loop (PyAudio / sounddevice) and probably a voice activity detector to chunk speech. Not in scope for a weekend notebook.